# Grafo pesato risolto con Algoritmo di Prim
Leggere i dati di un *grafo pesato* da un file, e memorizzarli in una matrice di adiacenza, dove ogni cella conterrà il peso del lato tra i vertici rappresentati dalle righe e colonne, altrimenti conterrà un valore che indichi l'assenza di un lato.

Adesso **risolviamo** un problema **con un algoritmo specifico** per il nostro problema non chiamo più `Pyomo`.

L'algoritmo può essere esatto o **euristico**, dipende dal problema che vogliamo risolvere.
Il nostro processo ora è *mettere insieme quello fatto le settimane scorse*.
## Cosa facciamo?
1. Apriamo un file, lo leggiamo, lo trasformiamo in una matrice di adiacenza.
2. Possiamo anche aggiungere che il nostro grafo sia pesato, quindi invece di memorizzare solo se esiste un lato tra i nodi, memorizziamo anche il peso di quel lato. In questo modo, la matrice di adiacenza conterrà i pesi dei lati tra i nodi, e se non esiste un lato, la cella conterrà un valore speciale (ad esempio, infinito o zero) per indicare l'assenza di un lato. Questo ci permetterà di risolvere problemi più complessi, come il problema del cammino minimo o il problema del flusso massimo, utilizzando algoritmi specifici per questi problemi.

Il nostro grafo **non è orientato**, quindi se esiste un lato tra i nodi `1` e `2`, allora esiste anche un lato tra i nodi `2` e `1`, e il peso di entrambi i lati sarà lo stesso.
```mermaid
graph LR
    n1(("1")) ---|3| n2(("2"))
    n2 ---|2| n3(("3"))
    n3 ---|10| n4(("4"))
    n3 ---|11| n5(("5"))
    n4 ---|7| n5
```
Nel nostro file ci sarà 


5 5 (Dimensione del grafo, numero di lati)

1 2 3 (lato tra i nodi 1 e 2 con peso 3)

2 3 2 

3 4 10 

3 5 11 

4 5 7 

che genera una matrice con questa forma
La matrice delle adiacenze $A = [a_{ij}]$ è:

$$
A =
\begin{pmatrix}
0 & 3 & 0 & 0 & 0 \\
3 & 0 & 2 & 0 & 0 \\
0 & 2 & 0 & 10 & 11 \\
0 & 0 & 10 & 0 & 7 \\
0 & 0 & 11 & 7 & 0 \\
\end{pmatrix}
$$

dove $a_{ij}$ rappresenta il peso dell’arco tra il nodo $i$ e il nodo $j$.
- $a_{ij} = 0$ se non esiste un arco tra $i$ e $j$
- $a_{ij} > 0$ è il peso dell’arco tra $i$ e $j$

In [1]:
import numpy as np

input= open("07 Grafo pesato.txt")
riga = input.readline()

#numero di vertici
n=int(riga.split()[0])

#numero di lati
m=int(riga.split()[1])

print("Numero di vertici, n =", n)
print("Numero di lati, m =", m)

lista=[]

infinito=99999
matrice=np.full((n,m),infinito,dtype=int)

for i in range(0, m):
    lista=input.readline().split()
    v1=int(lista[0])
    v2=int(lista[1])
    p=lista[2]
    matrice[v1-1][v2-1]=p
    matrice[v2-1][v1-1]=p
print(matrice)

Numero di vertici, n = 5
Numero di lati, m = 5
[[99999     3 99999 99999 99999]
 [    3 99999     2 99999 99999]
 [99999     2 99999    10    11]
 [99999 99999    10 99999     7]
 [99999 99999    11     7 99999]]


Ora vogliamo capire come collegare tutti i vertici di un grafo al costo minimo, cioè vogliamo trovare un *albero di copertura minimo*. 

- Un **albero di copertura** è un sottoinsieme dei lati del grafo che collega tutti i vertici senza formare cicli
- Il **costo minimo** è la somma dei pesi dei lati nell'albero di copertura. Per risolvere questo problema, possiamo utilizzare l'algoritmo di `Prim` o l'algoritmo di `Kruskal`, che sono algoritmi classici per trovare l'albero di copertura minimo in un grafo pesato. Questi algoritmi funzionano selezionando i lati con il peso più basso e aggiungendoli all'albero di copertura finché tutti i vertici sono collegati.

L'**algoritmo di Prim** è un metodo utilizzato per risolvere il problema dell'**Albero Ricoprente Minimo (Minimum Spanning Tree)** in un grafo non orientato con lati pesati. L'obiettivo è selezionare un sottoinsieme di lati che colleghi tutti i vertici del grafo senza formare cicli, minimizzando il costo totale dei lati scelti.

### Come funziona l'algoritmo
L'algoritmo segue una logica "greedy" (ingorda), costruendo l'albero un passo alla volta:
1.  **Partenza**: Si sceglie un vertice qualsiasi come punto di inizio (ad esempio il vertice 1), poiché il risultato finale non dipende dal punto di partenza.
2.  **Etichettatura**: Ogni vertice non ancora incluso nell'albero riceve un'etichetta che indica la sua distanza minima dall'albero parziale. All'inizio, i vicini del primo vertice avranno il peso del lato che li collega ad esso, mentre gli altri saranno a distanza infinita.
3.  **Espansione**: Ad ogni iterazione, si cerca il vertice fuori dall'albero che ha la **distanza minima** da uno dei vertici già inclusi.
4.  **Aggiornamento**: Una volta aggiunto un nuovo vertice all'albero, si controllano i suoi vicini rimasti fuori; se il nuovo vertice offre un cammino più breve per raggiungerli rispetto a quello precedentemente noto, si aggiornano le loro etichette e il loro "predecessore".
5.  **Termine**: Il processo si ripete finché tutti i vertici (N) non sono stati collegati utilizzando esattamente **N-1 lati**.

---

### Linea guida per risolvere l'esercizio (Implementazione)
Per implementare l'algoritmo in modo semplice, come suggerito nelle lezioni, puoi seguire questi passaggi:

1.  **Definisci una Classe `vertex`**: Crea un oggetto per ogni nodo che contenga informazioni fondamentali come l'**indice**, il **predecessore** (il nodo dell'albero a cui è collegato) e un valore booleano **`reached`** (raggiunto) per sapere se fa già parte dell'albero.
2.  **Prepara i Dati**: Utilizza una **matrice di adiacenza** per memorizzare i costi dei lati tra i vertici, leggendo i dati da un file.
3.  **Inizializza le Strutture**:
    *   Crea una lista che contenga tutti gli oggetti vertice.
    *   Imposta il primo vertice come "raggiunto" (`reached = True`).
    *   Usa un **contatore** per tenere traccia di quanti vertici hai aggiunto.
4.  **Ciclo Principale (`While`)**: Finché il contatore dei vertici raggiunti è minore del numero totale di vertici:
    *   **Ricerca del Minimo**: Esegui un ciclo su tutti i vertici non ancora raggiunti per trovare quello con la distanza minima dal proprio predecessore.
    *   **Aggiunta**: Segna il vertice trovato come raggiunto, incrementa il contatore e salva il lato corrispondente nella lista dei lati selezionati.
    *   **Aggiornamento dei Vicini**: Per ogni vertice non raggiunto, verifica se la distanza dal vertice appena aggiunto è minore della distanza dal suo attuale predecessore. In caso positivo, aggiorna il predecessore.

Questo approccio strutturato permette di gestire la complessità del grafo in modo elegante, riducendo la possibilità di errori durante l'aggiornamento delle distanze.

In [2]:
class vertex:
    def __init__(self):
      self.index=None
      self.name=None
      self.pred=None
      self.reached=False  #true se il vertice è in p

#inizializzazione
V=[]   #insieme dei vertici
for i in range(n):
  v=vertex()     #v è di tipo vertex
  v.name="v__"+str(i+1)
  v.index=i
  v.pred=0  #tutti inizialmente hanno predecessore 0
  V.append(v)

# Creo un insieme di lati E' che conterrà i lati dell'albero di copertura minimo
Eprime=[]

V[0].reached=True   #ho raggiunto il primo vertice
count_reached=1   #conta quanti vertici ho raggiunto
W=[]   #insieme dei vertici nell'albero

while (count_reached<n) :
 min=99999
 imin=None
#ciclo for in V, sto iterando sull'oggetto v vertice, i è l'indice. Potrei anche mettere i=0 fuori e poi lo incremento
 for i, v in enumerate(V):
  if v.reached==False:
    if matrice[v.pred][v.index]<min:
      min=matrice[v.pred][v.index]
      imin=i
 print("prossimo vertice:", imin+1)
 count_reached=count_reached+1
 vertice=V[imin]
 vertice.reached=True
 Eprime.append((vertice.pred, vertice.index))
 for w in V:
  if (w.reached==False) and matrice[vertice.index][w.index]<matrice[w.pred][w.index]:
    w.pred=vertice.index
print(Eprime)


prossimo vertice: 2
prossimo vertice: 3
prossimo vertice: 4
prossimo vertice: 5
[(0, 1), (1, 2), (2, 3), (3, 4)]
